In [0]:
%run ./opal_functions

In [0]:
cursor = ""
page_size = 1000
ret = get_opal_owners(cursor=cursor, page_size=page_size)
owners_accum = ret.json()['results']
if ret.json()['next'] != None:
    cursor = ret.json()['next']
    while cursor != None:
        ret = get_opal_owners(cursor=cursor, page_size=page_size)
        owners_accum += ret.json()['results']
        if ret.json()['next'] != 'None':
            cursor = ret.json()['next']

In [0]:
export = []
count = 1
for owner in owners_accum:
    print(f"{count} of {len(owners_accum)}: {owner['name']}")
    response = get_opal_owner_users(owner['owner_id'])
    for user in response.json()['users']:
        table = {}
        table['user_id'] = user['user_id']
        table['email'] = user['email']
        table['first_name'] = user['first_name']
        table['full_name'] = user['full_name']
        table['last_name'] = user['last_name']
        table['position'] = user['position']
        table['hr_idp_status'] = user['hr_idp_status']
        table['owner_id'] = owner['owner_id']
        export += [table]
    count = count + 1

In [0]:
import json
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, BooleanType, ArrayType

# Create a SparkSession
spark = SparkSession.builder.appName("StructTypeExample").getOrCreate()

schema = StructType([
    StructField("user_id", StringType(), True),
    StructField("email", StringType(), True),
    StructField("first_name", StringType(), True),
    StructField("full_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("position", StringType(), True),
    StructField("hr_idp_status", StringType(), True),
    StructField("owner_id", StringType(), True)
])


df = spark.createDataFrame(export, schema)

In [0]:
df.write.saveAsTable("workspace.opal_dev.opal_demo_owner_users", mode="overwrite") 